In [ ]:
# Problema: Integrar producción, disponibilidad y ambiente para analizar el desempeño diario de una fábrica.

import sqlite3
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
DATA = ROOT / "data"
DATABASE = ROOT / "temp" / "factory_operations.db"
OUTPUT = ROOT / "submission" / "factory_daily_performance.csv"
DATABASE.parent.mkdir(exist_ok=True)


In [ ]:
# Las fuentes tienen granos distintos: máquina-día, fábrica-día y empleado.

throughput = pd.read_csv(DATA / "machine_throughput_export.csv")
uptime = pd.read_csv(DATA / "machine_uptime_export.csv")
ambient = pd.read_csv(DATA / "factory_ambient_export.csv")
employees = pd.read_csv(DATA / "employee_metadata_export.csv")
throughput.shape, uptime.shape, ambient.shape, employees.shape


In [ ]:
# SQLite permite expresar y auditar la integración, sin convertir este taller en un curso de bases de datos.

DATABASE.unlink(missing_ok=True)
with sqlite3.connect(DATABASE) as connection:
    throughput.to_sql("throughput", connection, index=False)
    uptime.to_sql("uptime", connection, index=False)
    ambient.to_sql("ambient", connection, index=False)
    employees.to_sql("employees", connection, index=False)


In [ ]:
# Primero unimos las dos fuentes que comparten la llave máquina-día.

query = """
WITH machine_day AS (
    SELECT t.factory_id, t.machine_id, t.factory_date, t.daily_units_produced, u.hours_operational
    FROM throughput t
    INNER JOIN uptime u
        ON t.factory_id = u.factory_id
       AND t.machine_id = u.machine_id
       AND t.factory_date = u.factory_date
), factory_day AS (
    SELECT factory_id, factory_date,
           SUM(daily_units_produced) AS units_produced,
           ROUND(AVG(hours_operational), 2) AS average_hours_operational,
           COUNT(*) AS machines_reported
    FROM machine_day
    GROUP BY factory_id, factory_date
), employee_count AS (
    SELECT factory_id, COUNT(*) AS employees
    FROM employees
    GROUP BY factory_id
)
SELECT f.factory_id, f.factory_date, f.units_produced, f.average_hours_operational,
       f.machines_reported, e.employees, a.temp, a.humidity, a.pressure
FROM factory_day f
INNER JOIN ambient a ON f.factory_id = a.factory_id AND f.factory_date = a.date_measured
INNER JOIN employee_count e USING(factory_id)
ORDER BY f.factory_id, f.factory_date
"""
with sqlite3.connect(DATABASE) as connection:
    performance = pd.read_sql_query(query, connection)
performance.shape, performance.head()


In [ ]:
# La entrega tiene una fila por fábrica y día: el grano necesario para análisis posterior.

performance.to_csv(OUTPUT, index=False)
assert performance[["factory_id", "factory_date"]].duplicated().sum() == 0
assert performance.notna().all().all()
performance.groupby("factory_id")["units_produced"].mean().sort_values(ascending=False)
